# Notebook 11 — Model Training & Comparison

This notebook trains and compares six machine learning models (Linear Regression, Decision Tree, Random Forest, XGBoost, LightGBM, and CatBoost) to predict coconut yield, evaluating them across standard error and fit metrics.

## 1. Import Packages & Local Configuration

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Machine Learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Models Folder:", config.MODELS_DIR)

Models Folder: D:\A1\coco\Coconut_Yield_Prediction\models


## 2. Load Train and Test Datasets

In [2]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train.csv")
X_test = pd.read_csv(config.PROCESSED_DIR / "X_test.csv")
y_train = pd.read_csv(config.PROCESSED_DIR / "y_train.csv").values.ravel()
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").values.ravel()

print(f"Train observations: {X_train.shape[0]} | Features: {X_train.shape[1]}")
print(f"Test observations: {X_test.shape[0]}")

Train observations: 480 | Features: 39
Test observations: 128


## 3. Scale Features

Following strict ML ordering: scale numerical columns using a StandardScaler fit *only* on the training dataset, then apply the transform independently to train and test sets.

In [3]:
# All one-hot encoded columns represent binary indicators (0 or 1).
# We will scale only non-binary numerical features.
non_binary_cols = [col for col in X_train.columns if not X_train[col].isin([0, 1]).all()]
print("Numerical features to scale:", non_binary_cols)

scaler = StandardScaler()

# Scale train and test datasets separately
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[non_binary_cols] = scaler.fit_transform(X_train[non_binary_cols])
X_test_scaled[non_binary_cols] = scaler.transform(X_test[non_binary_cols])

# Save scaler for inference
joblib.dump(scaler, config.MODELS_DIR / "scaler.joblib")
print("[SAVED] Saved scaler to models/scaler.joblib")

Numerical features to scale: ['Rainfall', 'Temperature', 'Humidity', 'Soil_Moisture_Top', 'Soil_Moisture_Root', 'Soil_Moisture_Diff', 'Solar_Radiation', 'Temperature_Max', 'Temperature_Min', 'Temp_Range', 'Rainfall_Roll_3m', 'Rainfall_Roll_6m', 'Rainfall_Roll_12m', 'Soil_Moisture_Root_Roll_3m', 'Soil_Moisture_Root_Roll_6m', 'Rainfall_Anomaly', 'Temp_Anomaly', 'Soil_Moisture_Anomaly', 'Rainfall_Lag_3m', 'Rainfall_Lag_6m', 'Rainfall_Lag_12m', 'Rainfall_Lag_24m', 'Soil_Moisture_Root_Lag_12m', 'Soil_Moisture_Root_Lag_24m', 'Yield_Lag_1y', 'Yield_Lag_2y', 'Quarter', 'Month_Sin', 'Month_Cos', 'Rain_SoilMoist_Interaction', 'Temp_Humidity_Interaction', 'Light_Temp_Interaction']
[SAVED] Saved scaler to models/scaler.joblib


## 4. Train Models & Evaluate

We define a helper function to calculate MAPE (Mean Absolute Percentage Error):
$$MAPE = \frac{1}{n} \sum \left| \frac{y - \hat{y}}{y} \right| \times 100$$

In [4]:
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Instantiate models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.08, random_seed=42, verbosity=-1),
    "CatBoost": CatBoostRegressor(iterations=150, depth=5, learning_rate=0.08, random_seed=42, verbose=0)
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    
    # Predictions
    pred_train = model.predict(X_train_scaled)
    pred_test = model.predict(X_test_scaled)
    
    # Evaluation Metrics
    metrics = {
        "Model": name,
        "Train_MAE": mean_absolute_error(y_train, pred_train),
        "Test_MAE": mean_absolute_error(y_test, pred_test),
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, pred_train)),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, pred_test)),
        "Train_R2": r2_score(y_train, pred_train),
        "Test_R2": r2_score(y_test, pred_test),
        "Train_MAPE": mean_absolute_percentage_error(y_train, pred_train),
        "Test_MAPE": mean_absolute_percentage_error(y_test, pred_test)
    }
    results.append(metrics)

df_comparison = pd.DataFrame(results)
print("\nEvaluation Complete.")

Training Linear Regression...
Training Decision Tree...
Training Random Forest...
Training XGBoost...
Training LightGBM...
Training CatBoost...

Evaluation Complete.


## 5. Model Comparison Summary

In [6]:
# Display sorted comparison table by Test R2 score (highest first)
df_comparison = df_comparison.sort_values(by="Test_R2", ascending=False).reset_index(drop=True)
print(df_comparison.to_markdown(index=False))

# Save comparison results
df_comparison.to_csv(config.OUTPUTS_DIR / "model_comparison.csv", index=False)
print(f"\nSaved model comparison CSV to outputs/model_comparison.csv")

| Model             |   Train_MAE |   Test_MAE |   Train_RMSE |   Test_RMSE |   Train_R2 |   Test_R2 |   Train_MAPE |   Test_MAPE |
|:------------------|------------:|-----------:|-------------:|------------:|-----------:|----------:|-------------:|------------:|
| XGBoost           |    0.457138 |    3.0556  |     0.618163 |     6.63108 |   0.96335  | 0.119283  |      7.70025 |     32.9636 |
| CatBoost          |    1.02327  |    3.09805 |     1.27461  |     6.64548 |   0.844182 | 0.115454  |     16.656   |     34.3786 |
| Random Forest     |    0.722942 |    3.16752 |     0.910109 |     6.65891 |   0.920557 | 0.111875  |     11.9216  |     34.7407 |
| LightGBM          |    0.911962 |    3.24117 |     1.17966  |     6.79365 |   0.86653  | 0.0755676 |     14.4202  |     35.8871 |
| Linear Regression |    1.52689  |    3.48402 |     1.94694  |     6.9402  |   0.636443 | 0.0352564 |     24.4163  |     38.1312 |
| Decision Tree     |    1.22112  |    3.52138 |     1.61263  |     7.029   

## 6. Export Best Performing Model

In [7]:
best_model_name = df_comparison.loc[0, "Model"]
best_model = trained_models[best_model_name]

print(f"Best Model Identified: {best_model_name} (Test R²: {df_comparison.loc[0, 'Test_R2']:.4f})")

# Save best model checkpoint
joblib.dump(best_model, config.MODELS_DIR / "best_model.joblib")
print(f"Saved best model checkpoint to models/best_model.joblib")

Best Model Identified: XGBoost (Test R²: 0.1193)
Saved best model checkpoint to models/best_model.joblib
